### Search Engine With Tools And Agents

In [ ]:
## Arxiv--Research
## Tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [2]:
## Used the inbuilt tool of wikipedia
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [3]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [4]:
tools=[wiki,arxiv]

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
## Custom tools[RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

d:\Rajat\Installer\Anaconda\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Rajat\Installer\Anaconda\envs\myenv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [7]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A07A35EE40>, search_kwargs={})

In [8]:
from langchain_classic.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith-search","Search any information about Langsmith ")

retriever_tool.name

'langsmith-search'

In [9]:
tools=[wiki,arxiv,retriever_tool]

In [10]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\Rajat\\Installer\\Anaconda\\envs\\myenv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith ', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001A07A138B80>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001A07A13AC00>)]

In [23]:
## Run all this tools with Agents and LLM Models

## Tools, LLM-->AgentExecutor
from langchain_groq import ChatGroq
from dotenv import load_dotenv
# import openai
load_dotenv()
import os

groq_api_key=os.getenv("GROQ_API_KEY")
# openai.api_key=os.getenv("OPENAI_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b")

In [24]:
## Prompt Template
from langchain_classic.hub import pull
prompt=pull("hwchase17/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [25]:
## Agents
from langchain_classic.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm,tools,prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [26]:
## Agent Executer
from langchain_classic.agents import AgentExecutor
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [30]:
agent_executor.invoke({"input":"Tell me about GenAI in detail"})



> Entering new AgentExecutor chain...
## Generative AI (GenAI) – A Detailed Overview  

### 1. What is Generative AI?  
**Generative AI** refers to a class of artificial‑intelligence systems that can create new content—text, images, audio, video, code, or even 3‑D models—rather than merely classifying or retrieving existing data. The hallmark of these systems is **generation**: given a prompt or conditioning signal, they produce an output that did not exist before, often indistinguishable from human‑made artifacts.

| Aspect | Traditional AI / Discriminative | Generative AI |
|--------|--------------------------------|---------------|
| Goal   | Predict a label / decision (e.g., “Is this email spam?”) | Produce new data (e.g., “Write a poem about sunrise”) |
| Training Objective | Maximize likelihood of correct class | Model the *distribution* of the data so that sampling yields realistic samples |
| Typical Models | Logistic regression, SVM, CNNs for classification | Autoregressive 

{'input': 'Tell me about GenAI in detail',
 'output': '## Generative AI (GenAI) – A Detailed Overview  \n\n### 1. What is Generative AI?  \n**Generative AI** refers to a class of artificial‑intelligence systems that can create new content—text, images, audio, video, code, or even 3‑D models—rather than merely classifying or retrieving existing data. The hallmark of these systems is **generation**: given a prompt or conditioning signal, they produce an output that did not exist before, often indistinguishable from human‑made artifacts.\n\n| Aspect | Traditional AI / Discriminative | Generative AI |\n|--------|--------------------------------|---------------|\n| Goal   | Predict a label / decision (e.g., “Is this email spam?”) | Produce new data (e.g., “Write a poem about sunrise”) |\n| Training Objective | Maximize likelihood of correct class | Model the *distribution* of the data so that sampling yields realistic samples |\n| Typical Models | Logistic regression, SVM, CNNs for classifi

In [31]:
agent_executor.invoke({"input":"What is machine learning"})



> Entering new AgentExecutor chain...
**Machine learning (ML)** is a subfield of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn patterns and make decisions from data, without being explicitly programmed for each specific task.

### Core Idea
- **Learning from data**: Instead of hard‑coding rules, an ML system infers a function \(f\) that maps inputs \(X\) to outputs \(Y\) by analyzing examples \(\{(x_i, y_i)\}\).
- **Generalization**: The goal is for the learned model to perform well on new, unseen data, not just the training examples.

### Main Categories
| Category | Typical Tasks | Common Algorithms |
|----------|---------------|-------------------|
| **Supervised learning** | Classification (e.g., spam detection), Regression (e.g., price prediction) | Linear regression, logistic regression, decision trees, random forests, support‑vector machines, neural networks |
| **Unsupervised learning** | Clustering, dimens

{'input': 'What is machine learning',
 'output': '**Machine learning (ML)** is a subfield of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn patterns and make decisions from data, without being explicitly programmed for each specific task.\n\n### Core Idea\n- **Learning from data**: Instead of hard‑coding rules, an ML system infers a function\u202f\\(f\\) that maps inputs\u202f\\(X\\) to outputs\u202f\\(Y\\) by analyzing examples \\(\\{(x_i, y_i)\\}\\).\n- **Generalization**: The goal is for the learned model to perform well on new, unseen data, not just the training examples.\n\n### Main Categories\n| Category | Typical Tasks | Common Algorithms |\n|----------|---------------|-------------------|\n| **Supervised learning** | Classification (e.g., spam detection), Regression (e.g., price prediction) | Linear regression, logistic regression, decision trees, random forests, support‑vector machines, neural networks |\n| *

In [ ]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c
Invoking: `wikipedia` with `{'query': 'Attention Is All You Need'}`


Page: Attention Is All You Need
Summary: "Attention Is All You Need" is a 2017 landmark research paper authored by eight scientists working at Google, that introduced a new deep learning architecture known as the transformer based on attention mechanThe Transformer is a neural network architecture that was first introduced in this paper. It's primarily used for natural language processing tasks, such as machine translation, text summarization, and language modeling. The Transformer architecture is based on self-attention mechanisms, which allow it to focus on specific parts of the

{'input': "What's the paper 1706.03762 about?",
 'output': "The Transformer is a neural network architecture that was first introduced in this paper. It's primarily used for natural language processing tasks, such as machine translation, text summarization, and language modeling. The Transformer architecture is based on self-attention mechanisms, which allow it to focus on specific parts of the input data while processing it. This architecture has been widely adopted in many NLP tasks and has achieved state-of-the-art results in many applications."}